# NB4: Replay Only
## Continual learning experiments (continued)
**GPU required.** You can run this notebook at the same time as NB2 if you want.

This notebook runs three continual learning setups. Later you compare them to the baselines from NB2:

- **`replay_only`:** Train in order with a **replay buffer** only (no EWC). A sample of past examples gets mixed into later training so the optimizer still sees old tasks.

### Inputs (Kaggle: add these as input datasets)
- **`pi-detection-data`** at `/kaggle/input/pi-detection-data/` (parquet files from NB1)
- **`pi-detection-utils`** at `/kaggle/input/pi-detection-utils/` (`utils.py` from your repo)
- **`pi-detection-baselines`** at `/kaggle/input/pi-detection-baselines/` *(optional: attach NB2 output so the big table can include `static_joint` and `naive_sequential`)*

### Outputs (save as a Kaggle dataset, e.g. `pi-detection-cl`)
```
/kaggle/working/
  results/results_cl.json
  checkpoints/ewc_only_after_*.pt
  checkpoints/replay_only_after_*.pt
  checkpoints/ewc_plus_replay_after_*.pt   (NB4 uses these checkpoints)
  replay_buffer/ewc_plus_replay.json         (saved replay buffer)
```

## Install & Setup

In [1]:
!pip install -q transformers accelerate scikit-learn

import sys, os
sys.path.append('/kaggle/input/datasets/dabiraomotoso/pi-detection-utils')
from utils import *

os.makedirs(CFG['checkpoint_dir'], exist_ok=True)
os.makedirs(CFG['results_dir'],    exist_ok=True)
os.makedirs(CFG['replay_dir'],     exist_ok=True)

print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


## 1. Load data and build dataloaders
Read the three parquet files from NB1 when they are present. For each task we build **train**, **validation**, and **test** loaders using the split and batch settings in **`utils.CFG`** (same idea as NB2). Missing files are skipped with a message. You generally want all three tasks loaded before you run the experiments below.

In [2]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
login(secret_value_0)

In [3]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
data_dir  = '/kaggle/input/datasets/dabiraomotoso/pi-detection-data'
tasks     = {}

for task_name, fname in [
    ('T1_LLMail',      't1_llmail.parquet'),
    ('T2_HackAPrompt', 't2_hackaprompt.parquet'),
    ('T3_BIPIA',       't3_bipia.parquet'),
]:
    path = os.path.join(data_dir, fname)
    if os.path.exists(path):
        df = pd.read_parquet(path)
        tr, va, te = make_loaders(df, tokenizer)
        tasks[task_name] = {'train': tr, 'val': va, 'test': te, 'df': df}
        print(f'  {task_name}: {len(tr.dataset)} train | {len(va.dataset)} val | {len(te.dataset)} test')
    else:
        print(f'  {task_name}: FILE NOT FOUND — skipping')

print(f'Loaded {len(tasks)} tasks.')

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

  T1_LLMail: 14000 train | 3000 val | 3000 test
  T2_HackAPrompt: 14000 train | 3000 val | 3000 test
  T3_BIPIA: 14000 train | 3000 val | 3000 test
Loaded 3 tasks.


In [4]:
import gc, os
import utils as _utils

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

# Patch 1: flush GPU after checkpoint save
original_save = _utils.save_checkpoint

def save_checkpoint_and_flush(model, path):
    original_save(model, path)
    gc.collect()
    torch.cuda.empty_cache()
    free = (torch.cuda.get_device_properties(0).total_memory
            - torch.cuda.memory_allocated()) / 1e9
    print(f'  GPU cache cleared. Free VRAM: {free:.2f}GB')

_utils.save_checkpoint = save_checkpoint_and_flush

# Patch 2: enable gradient checkpointing on load_model to halve activation memory
original_load_model = _utils.load_model

def load_model_with_gc():
    model = original_load_model()
    model.gradient_checkpointing_enable()
    print('  Gradient checkpointing enabled.')
    return model

_utils.load_model = load_model_with_gc

# Patch 3: patch run_experiment to clear GPU between tasks
# (must patch run_experiment itself since it holds a direct ref to train_task)
original_run_experiment = _utils.run_experiment

def run_experiment_with_cleanup(experiment_name, tasks, tokenizer,
                                 use_ewc=False, use_replay=False, joint_training=False):
    import types

    # Wrap train_task inside a closure that clears GPU after each task
    original_tt = _utils.train_task

    def train_task_patched(*args, **kwargs):
        result = original_tt(*args, **kwargs)
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        free = (torch.cuda.get_device_properties(0).total_memory
                - torch.cuda.memory_allocated()) / 1e9
        print(f'  Inter-task GPU cleared. Free VRAM: {free:.2f}GB')
        return result

    # Temporarily replace train_task in the utils module so run_experiment sees it
    _utils.train_task = train_task_patched
    try:
        result = original_run_experiment(
            experiment_name, tasks, tokenizer,
            use_ewc=use_ewc, use_replay=use_replay, joint_training=joint_training
        )
    finally:
        _utils.train_task = original_tt  # restore

    return result

_utils.run_experiment = run_experiment_with_cleanup

# Also patch the name in the notebook's global namespace
run_experiment = run_experiment_with_cleanup

print('All patches active.')

All patches active.


## 2. Experiment 4: Replay only
**Goal:** Same **T1, then T2, then T3** order, but use **experience replay** only (no EWC). After each task, a chunk of that task’s data is added to a **replay buffer** (max size and mix fraction live in **`utils.CFG`**, often around 10% replay in each batch). Later training batches blend new-task data with replayed old rows so gradients still touch past tasks.

This is **data-level** memory, not a weight penalty. Compare the table to **EWC only** and to **naive sequential** from NB2.

In [5]:
all_results = {}  # Fix the NameError

all_results['replay_only'], _model = run_experiment(
    'replay_only', tasks, tokenizer,
    use_ewc=False, use_replay=True, joint_training=False,
)
del _model  # free the returned model from GPU
gc.collect()
torch.cuda.empty_cache()

save_results(all_results, f'{CFG["results_dir"]}/results_cl_replay.json')


EXPERIMENT: replay_only
  EWC: False | Replay: True | Joint: False


`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.weight      

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Gradient checkpointing enabled.

--- Training on T1_LLMail (step 1/3) ---
  Epoch 1/3 | Train Loss: 0.1562 | Val F1: 0.9845 | Val Loss: 0.0702
  Epoch 2/3 | Train Loss: 0.0500 | Val F1: 0.9905 | Val Loss: 0.0471
  Epoch 3/3 | Train Loss: 0.0329 | Val F1: 0.9908 | Val Loss: 0.0449
  Best val F1: 0.9908
  Inter-task GPU cleared. Free VRAM: 14.14GB
  Replay buffer: +5000 from T1_LLMail. Total: 5000
  After T1_LLMail → T1_LLMail test F1: 0.9929
  Checkpoint saved: /kaggle/working/checkpoints/replay_only_after_T1_LLMail.pt
  GPU cache cleared. Free VRAM: 14.14GB

--- Training on T2_HackAPrompt (step 2/3) ---
  Epoch 1/3 | Train Loss: 0.7754 | Val F1: 0.7217 | Val Loss: 0.5596
  Epoch 2/3 | Train Loss: 0.5785 | Val F1: 0.7405 | Val Loss: 0.5497
  Epoch 3/3 | Train Loss: 0.5140 | Val F1: 0.7598 | Val Loss: 0.5336
  Best val F1: 0.7598
  Inter-task GPU cleared. Free VRAM: 14.14GB
  Replay buffer: +2500 from T2_HackAPrompt. Total: 5000
  After T2_HackAPrompt → T1_LLMail test F1: 0.9194
  Afte